In [0]:
import json
import requests
from datetime import datetime, timezone
from pyspark.sql import functions as F

pit_url = "https://api.jolpi.ca/ergast/f1/current/last/pitstops.json"

pages = []
offset = 0
page_size = 1000

while True:
    response = requests.get(
        pit_url,
        params={
            "limit": page_size,
            "offset": offset
        },
        timeout=30
    )

    response.raise_for_status()

    page = response.json()
    pages.append(page)

    metadata = page["MRData"]

    total_records = int(metadata["total"])
    returned_records = int(metadata["limit"])

    offset += returned_records

    if offset >= total_records or returned_records == 0:
        break

print("Number of API pages:", len(pages))
print("Total pit-stop records:", total_records)

In [0]:
first_race = pages[0]["MRData"]["RaceTable"]["Races"][0]

season = int(first_race["season"])
round_number = int(first_race["round"])
ingestion_time = datetime.now(timezone.utc)

bronze_pit_rows = [
    (
        "jolpica",
        "pit_stops",
        season,
        round_number,
        json.dumps(page),
        pit_url,
        ingestion_time
    )
    for page in pages
]

bronze_pit_columns = [
    "source_name",
    "entity_type",
    "season",
    "round",
    "payload_json",
    "source_url",
    "ingested_at"
]

bronze_pit_df = spark.createDataFrame(
    bronze_pit_rows,
    bronze_pit_columns
)

catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

bronze_table = f"{catalog}.f1_bronze.api_responses"

(
    bronze_pit_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_table)
)

print("Raw pit-stop data saved to Bronze.")

In [0]:
display(
    spark.table(bronze_table)
    .filter(F.col("entity_type") == "pit_stops")
    .select(
        "season",
        "round",
        "entity_type",
        "source_url",
        "ingested_at"
    )
)